In [ ]:
#%cd ..

c:\Users\coffe\Desktop\titanic-ml\titanic-ml


In [7]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold

from src.ensemble_validation import get_lgbm_oof

SEED = 42

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
train = pd.read_csv("data/train.csv")

X = train.drop(columns="Survived")
y = train["Survived"]

In [9]:
best_lgbm_params = {
    "learning_rate": 0.022,
    "max_depth": 7,
    "num_leaves": 44,
    "min_child_samples": 21,
    "subsample": 1,
    "colsample_bytree": 0.707,
    "reg_alpha": 0.245,
    "reg_lambda": 0.002,
    "n_estimators": 500,
    "subsample_freq": 1,
}

oof_lgbm = get_lgbm_oof(
    X=X,
    y=y, 
    model_params=best_lgbm_params, 
    n_splits=5, 
    seed=SEED
)

oof_lgbm.shape


(891,)

In [10]:
from sklearn.metrics import accuracy_score

oof_lgbm_pred = (oof_lgbm >= 0.5).astype(int)

lgbm_oof_accuracy = accuracy_score(
    y,
    oof_lgbm_pred,
)

print("LightGBM OOF accuracy:", lgbm_oof_accuracy)

LightGBM OOF accuracy: 0.8484848484848485


In [11]:
from src.ensemble_validation import get_catboost_oof

In [13]:
best_catboost_params = {
    "depth": 6,
    "learning_rate": 0.105,
    "l2_leaf_reg": 2.183,
    "random_strength": 2.571,
}

oof_catboost = get_catboost_oof(
    X=X,
    y=y,
    model_params=best_catboost_params,
    cat_features=['Sex', 'Embarked', 'Pclass'],
    n_splits=5,
    seed=SEED,
)

catboost_oof_pred = (oof_catboost >= 0.5).astype(int)

catboost_oof_accuracy  = accuracy_score(
    y,
    catboost_oof_pred
)

print(
    "CatBoost OOF accuracy:",
    catboost_oof_accuracy
)

CatBoost OOF accuracy: 0.8327721661054994


In [15]:
from src.ensemble_validation import get_dnn_oof

In [16]:
oof_dnn = get_dnn_oof(
    X=X,
    y=y,
    n_splits=5,
    epochs=100,
    batch_size=32,
    lr=3e-3,
    seed=SEED
)

dnn_oof_pred = (oof_dnn >= 0.5).astype(int)

dnn_oof_accuracy = accuracy_score(
    y, 
    dnn_oof_pred
)

print("DNN OOF accuracy:", dnn_oof_accuracy)

DNN OOF accuracy: 0.835016835016835
